# SIH26175 DepthWizard - GAMUS Fine-Tuning on Kaggle GPU
### Fine-Tuning Depth Anything V2 Base on 5,000 GAMUS Tiles with Tall-Building Weighted Loss

This notebook trains Depth Anything V2 on the official `earthflow/GAMUS` dataset to minimize LiDAR height error down toward 1 meter.

In [ ]:
# 1. Install Dependencies
!pip install -q transformers torch torchvision datasets h5py rasterio scipy

In [ ]:
# 2. Verify GPU Acceleration
import torch
print(f"Using GPU: {torch.cuda.get_device_name(0)}")
assert torch.cuda.is_available(), "Please enable GPU accelerator (T4 or P100) in Kaggle settings!"

In [ ]:
# 3. Custom Tall-Building Weighted Loss
import torch.nn as nn

class TallBuildingWeightedLoss(nn.Module):
    """
    Combines Scale-Invariant Logarithmic (SILog) loss with an exponential height weight
    to penalize errors on commercial and tall buildings (>15m) up to 3x more heavily.
    """
    def __init__(self, alpha=3.0, lambda_tall=2.0):
        super().__init__()
        self.alpha = alpha
        self.lambda_tall = lambda_tall

    def forward(self, pred, target, mask):
        valid = (mask > 0) & (target > 0.1) & (pred > 0.1)
        if not torch.any(valid):
            return torch.tensor(0.0, device=pred.device, requires_grad=True)
        
        p, t = pred[valid], target[valid]
        d = torch.log(p) - torch.log(t)
        silog = torch.sqrt(torch.mean(d ** 2) - 0.5 * (torch.mean(d) ** 2))
        
        # Tall building exponential weight
        weights = 1.0 + self.alpha * torch.clamp(t / 30.0, 0.0, 2.5)
        tall_l1 = torch.mean(weights * torch.abs(p - t))
        
        return silog + self.lambda_tall * tall_l1

In [ ]:
# 4. Load Pre-trained Depth Anything V2 Base
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

model_id = "depth-anything/Depth-Anything-V2-Base-hf"
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModelForDepthEstimation.from_pretrained(model_id).cuda()
print("Model loaded successfully!")

In [ ]:
# 5. Training Loop with Mixed Precision & Cosine Annealing
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-2)
scaler = torch.cuda.amp.GradScaler()
criterion = TallBuildingWeightedLoss()

print("Ready to train on GAMUS! Save checkpoint to depthwizard_gamus_v2.pt")